# Track quarterly cohorts

*Data Analysis in Python · Topic 11.2*

Your CRM team plans its win-back emails by quarter, and they want to know how many of a quarter's new customers order again in the next one, and whether that depends on the quarter they started in. Build a quarterly cohort table, and average the first quarter after the start across the cohorts.

**Before you start**, choose **File → Save a copy in Drive**, and work in the copy that opens. This workbook is shared with every student, so Colab doesn't save your changes to it.

1. Run the parameters cell and the cell that reads the `customers` and `orders` tables, with the orders before `CUTOFF`.
2. In the first cell marked ✏️, create the `new_customers` table: the customers created from `START` to before `CUTOFF`, with a `cohort` column, the quarter of `created_at`. Then create `cohort_sizes`: the number of customers in each cohort.
3. In the second cell marked ✏️, create the `cohort_orders` table: the orders of the customers in the `new_customers` table, with the customer's `created_at` as `created_at_customer` and the `cohort`. Add `quarters_since`: the calendar quarters from the customer's `created_at` to the order's, 0 for the quarter the customer started in.
4. In the last cell marked ✏️, create the `retention` table: one row per cohort and one column per `quarters_since`, with the share of the cohort's customers who placed an order in that quarter. Then create `quarter_1_retention`: the share of customers who ordered in their quarter 1, over the cohorts that have had a quarter 1, calculated from the totals.
5. Choose **Runtime → Restart session and run all**, and read the **Check your work** cell's output.

**Expected output**: The `new_customers` table has 40,126 rows, in 8 cohorts from 4,109 customers in 2024Q1 to 7,617 in 2025Q4. The `retention` table is 8 × 8, and 18.1% of 2024Q3's customers ordered again in their quarter 1. `quarter_1_retention` is about `0.1162`.

## Parameters

In [ ]:
import pandas as pd

DATA_URL = "https://storage.googleapis.com/simmer-python-course-data/simmer-store/v1/"
START = "2024-01-01"
CUTOFF = "2026-01-01"  # The timestamps are in UTC: the orders before it are the orders of 2024 and 2025.

In [ ]:
# The customers and their orders, without the orders placed after the data ends. Run it, and leave it as it is.
customers = pd.read_csv(DATA_URL + "customers.csv", parse_dates=["created_at"])
orders = pd.read_csv(DATA_URL + "orders.csv", parse_dates=["created_at"],
                     usecols=["order_id", "customer_id", "status", "subtotal", "coupon", "created_at"])
orders = orders[orders["created_at"] < CUTOFF]
customers.shape, orders.shape

## The cohorts

In [ ]:
new_customers = customers[(customers["created_at"] >= START) & (customers["created_at"] < CUTOFF)].copy()
new_customers["cohort"] = new_customers["created_at"].dt.to_period("Q")
cohort_sizes = new_customers.groupby("cohort").size()
cohort_sizes

## Quarters since the start

In [ ]:
cohort_orders = orders.merge(new_customers[["customer_id", "created_at", "cohort"]], on="customer_id",
                             suffixes=("", "_customer"), validate="many_to_one")
cohort_orders["quarters_since"] = ((cohort_orders["created_at"].dt.year - cohort_orders["created_at_customer"].dt.year) * 4
                                   + cohort_orders["created_at"].dt.quarter - cohort_orders["created_at_customer"].dt.quarter)
len(cohort_orders), cohort_orders["quarters_since"].max()

## The retention table

In [ ]:
active = cohort_orders.pivot_table(index="cohort", columns="quarters_since", values="customer_id", aggfunc="nunique")
retention = active.div(cohort_sizes, axis=0)
quarter_1_retention = active[1].sum() / cohort_sizes[active[1].notna()].sum()
retention.round(3), quarter_1_retention

In [ ]:
# @title Check your work
# @markdown This cell runs last, with the others. Run it with Runtime → Restart session and run all.
# This cell only reads your variables and the cells that have run. It doesn't change anything.
def check_your_work():
    import sys
    import pandas as pd
    found = globals()
    results = []

    inputs = found.get("In")
    history = [source for source in inputs[1:-1] if "def check_your_work" not in source] if isinstance(inputs, list) else None
    checked_before = isinstance(inputs, list) and any("def check_your_work" in source for source in inputs[1:-1])
    if history is None:
        results.append((None, "This check can't see which cells have run, so it can't tell whether this was a clean run. Choose Runtime → Restart session and run all to be sure."))
    else:
        repeated = [source for source in history if source.strip() and history.count(source) > 1]
        error = getattr(sys, "last_exc", None) or getattr(sys, "last_value", None)
        if checked_before:
            results.append((None, "This cell has run before in this session, so the cells above may have changed since they ran. Choose Runtime → Restart session and run all to check the notebook as it is now."))
        elif repeated:
            first_line = repeated[0].strip().splitlines()[0]
            results.append((None, f"Some cells ran more than once in this session, such as the one that starts with {first_line!r}. Choose Runtime → Restart session and run all, so every cell runs once, from the top."))
        elif error is not None:
            results.append((None, f"A cell stopped with {type(error).__name__} earlier in this session, so the cells after it may not have run in order. Choose Runtime → Restart session and run all."))
        else:
            results.append((True, "Every cell in this session ran once, with no errors."))

    def number(value):
        return pd.api.types.is_number(value) and not isinstance(value, (pd.DataFrame, pd.Series))

    def near(value, *targets, tolerance=0.00001):
        return number(value) and any(abs(value - target) < tolerance for target in targets)

    new_customers, sizes = found.get("new_customers"), found.get("cohort_sizes")
    expected_sizes = [4109, 3436, 4155, 5688, 5296, 4653, 5172, 7617]
    if not isinstance(new_customers, pd.DataFrame) or "cohort" not in new_customers.columns:
        results.append((False, "new_customers should be a DataFrame with a cohort column: the customers created from START to before CUTOFF."))
    elif len(new_customers) in (53272, 53251):
        results.append((False, f"The new_customers table has {len(new_customers):,} rows: it still has the 13,125 customers from 2023, whose first orders aren't in the file. Keep the customers created from START."))
    elif len(new_customers) != 40126:
        results.append((False, f"The new_customers table should have the 40,126 customers created from START to before CUTOFF. It has {len(new_customers):,}."))
    elif not isinstance(sizes, pd.Series) or len(sizes) == 24:
        results.append((False, "cohort_sizes should be a Series with one number per quarter: 8 of them. If it has 24, the cohorts are months: use .dt.to_period(\"Q\")."))
    elif [int(size) for size in sizes] != expected_sizes:
        results.append((False, f"cohort_sizes should be {', '.join(f'{n:,}' for n in expected_sizes)}, from 2024Q1 to 2025Q4. It is {', '.join(f'{int(n):,}' for n in sizes)}."))
    else:
        results.append((True, "40,126 new customers in 8 quarterly cohorts, from 4,109 in 2024Q1 to 7,617 in 2025Q4."))

    cohort_orders = found.get("cohort_orders")
    if not isinstance(cohort_orders, pd.DataFrame) or "quarters_since" not in cohort_orders.columns:
        results.append((False, "cohort_orders should be a DataFrame with a quarters_since column: the new customers' orders."))
    elif len(cohort_orders) != 57595:
        results.append((False, f"The cohort_orders table should have the new customers' 57,595 orders: an inner join of orders and new_customers. It has {len(cohort_orders):,}."))
    elif cohort_orders["quarters_since"].min() != 0 or cohort_orders["quarters_since"].max() != 7:
        results.append((False, f"quarters_since should run from 0, the quarter the customer started in, to 7. It runs from {cohort_orders['quarters_since'].min()} to {cohort_orders['quarters_since'].max()}: count it as (year difference) * 4 + (quarter difference)."))
    elif (cohort_orders["quarters_since"] == 1).sum() != 4677:
        results.append((False, "quarters_since should count calendar quarters: an order in the quarter after the customer's start is 1, even if only a week has passed. Use .dt.year and .dt.quarter, not the days or months between the dates."))
    else:
        results.append((True, "57,595 orders, each with the calendar quarters since its customer's start."))

    retention = found.get("retention")
    cells = {("2024Q1", 0): 0.995619, ("2024Q1", 1): 0.085179, ("2024Q3", 1): 0.181227, ("2024Q4", 4): 0.126055, ("2025Q3", 1): 0.180201, ("2024Q1", 7): 0.112436}
    if not isinstance(retention, pd.DataFrame):
        results.append((False, "retention should be a DataFrame: one row per cohort, one column per quarters_since."))
    elif retention.shape != (8, 8):
        results.append((False, f"retention should be 8 × 8: 8 cohorts and quarters 0 to 7. It is {retention.shape[0]} × {retention.shape[1]}."))
    else:
        index = [str(label) for label in retention.index]
        got = {cell: retention.iloc[index.index(cell[0]), list(retention.columns).index(cell[1])] if cell[0] in index and cell[1] in list(retention.columns) else None for cell in cells}
        if got[("2024Q3", 1)] is not None and near(got[("2024Q3", 1)], 0.224067):
            results.append((False, "retention for 2024Q3 in quarter 1 is 0.2241: it counts orders, not customers. Count each customer once with aggfunc=\"nunique\"."))
        elif any(value is None or not near(value, cells[cell]) for cell, value in got.items()):
            cell = next(cell for cell, value in got.items() if value is None or not near(value, cells[cell]))
            results.append((False, f"retention for {cell[0]} in quarter {cell[1]} should be about {cells[cell]:.4f}: the cohort's customers with an order in that quarter, divided by the cohort's size. It is {got[cell]!r}."))
        elif not pd.isna(retention.iloc[7, 1]):
            results.append((False, "retention for 2025Q4 in quarter 1 should be empty: that quarter hasn't happened yet."))
        else:
            results.append((True, "18.1% of the customers who started in Q3 2024 ordered again in Q4, and 18.0% of Q3 2025's: the Q3 cohorts come back for the holidays."))

    q1 = found.get("quarter_1_retention")
    if near(q1, 0.094128):
        results.append((False, "quarter_1_retention is 0.0941: it counts 2025Q4's cohort, which hasn't had a quarter 1 yet, as 0%. Leave it out of both sums."))
    elif near(q1, 0.116111):
        results.append((False, "quarter_1_retention is 0.1161: the mean of the cohorts' rates. Calculate it from the totals, so each cohort counts by its size."))
    elif near(q1, 0.143868):
        results.append((False, "quarter_1_retention is 0.1439: it counts orders, not customers."))
    elif not near(q1, 0.116183):
        results.append((False, f"quarter_1_retention should be the customers active in quarter 1, summed over the cohorts that have had one, divided by those cohorts' customers: about 0.1162. It is {q1!r}."))
    else:
        results.append((True, f"{q1:.1%} of new customers order again in the quarter after they start."))

    for passed, message in results:
        print({True: "✅ ", False: "❌ ", None: "⚠️ "}[passed] + message)
    if any(passed is False for passed, message in results):
        print("\nFix the first ❌, then choose Runtime → Restart session and run all.")
    elif any(passed is None for passed, message in results):
        print("\nYour results match. Restart the session and run all, to check them in a clean run.")
    else:
        print("\nEverything matches, in a clean run.")


check_your_work()

<details>
<summary><b>Show solution</b></summary>

Try the exercise yourself first. This is one way to solve it: if the check passes, your way is right too.

```python
new_customers = customers[(customers["created_at"] >= START) & (customers["created_at"] < CUTOFF)].copy()
new_customers["cohort"] = new_customers["created_at"].dt.to_period("Q")
cohort_sizes = new_customers.groupby("cohort").size()
cohort_sizes
```

```python
cohort_orders = orders.merge(new_customers[["customer_id", "created_at", "cohort"]], on="customer_id",
                             suffixes=("", "_customer"), validate="many_to_one")
cohort_orders["quarters_since"] = ((cohort_orders["created_at"].dt.year - cohort_orders["created_at_customer"].dt.year) * 4
                                   + cohort_orders["created_at"].dt.quarter - cohort_orders["created_at_customer"].dt.quarter)
len(cohort_orders), cohort_orders["quarters_since"].max()
```

```python
active = cohort_orders.pivot_table(index="cohort", columns="quarters_since", values="customer_id", aggfunc="nunique")
retention = active.div(cohort_sizes, axis=0)
quarter_1_retention = active[1].sum() / cohort_sizes[active[1].notna()].sum()
retention.round(3), quarter_1_retention
```

</details>